# Spotify Hit Prediction

## Objective

Can we predict whether a Spotify track belongs to the top 20% of popularity using its audio characteristics and explicit-content metadata?

We will use PySpark and Spark MLlib to train and compare Logistic Regression, Random Forest and Gradient-Boosted Trees classifiers.

The test set will remain separate from model training and preprocessing.

In [5]:
# import and spark session
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Spotify Hit Prediction")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

In [9]:
#load data

data_path = "../data/processed/spotify_tracks_ml.csv"

clean_df = spark.read.csv(data_path, header=True, inferSchema=True)

print("Rows:", clean_df.count())
print("Columns:", len(clean_df.columns))

clean_df.printSchema()
clean_df.show(5, truncate=False)

Rows: 89740
Columns: 17
root
 |-- track_id: string (nullable = true)
 |-- popularity: double (nullable = true)
 |-- danceability: double (nullable = true)
 |-- energy: double (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- liveness: double (nullable = true)
 |-- valence: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-- loudness: double (nullable = true)
 |-- duration_ms: double (nullable = true)
 |-- key: double (nullable = true)
 |-- mode: double (nullable = true)
 |-- time_signature: double (nullable = true)
 |-- explicit: boolean (nullable = true)
 |-- hit: integer (nullable = true)

+----------------------+----------+------------+------+-----------+------------+----------------+--------+-------+-------+--------+-----------+---+----+--------------+--------+---+
|track_id              |popularity|danceability|energy|speechiness|acousticness|instrumentalness|

In [10]:
print("Unique track IDs:",
      clean_df.select("track_id").distinct().count())

print("Missing track IDs:",
      clean_df.filter(
          F.col("track_id").isNull() |
          (F.trim(F.col("track_id")) == "")
      ).count())

print("Missing popularity:",
      clean_df.filter(
          F.col("popularity").isNull()
      ).count())

Unique track IDs: 89740
Missing track IDs: 0
Missing popularity: 0


In [11]:
audio_features = [
    "danceability",
    "energy",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "loudness",
    "duration_ms",
    "key",
    "mode",
    "time_signature"
]

feature_columns = audio_features + ["explicit"]

In [ ]:
#even though already cleaned add code just in case

In [13]:
#clean the data to remove rows with missing track_id or popularity
model_source = clean_df.filter(
    F.col("track_id").isNotNull() &
    (F.trim(F.col("track_id")) != "") &
    F.col("popularity").isNotNull()
)

In [14]:
# aggregate 1 row per track_id, averaging popularity and audio features, and taking max of explicit
track_df = (
    model_source
    .groupBy("track_id")
    .agg(
        F.avg("popularity").alias("popularity"),

        *[
            F.avg(c).alias(c)
            for c in audio_features
        ],

        F.max(
            F.col("explicit").cast("int")
        ).alias("explicit")
    )
)

print("Rows:", track_df.count())
print("Unique tracks:",
      track_df.select("track_id").distinct().count())

track_df.show(5, truncate=False)

Rows: 89740
Unique tracks: 89740
+----------------------+----------+------------+------+-----------+------------+----------------+--------+-------+-------+--------+-----------+---+----+--------------+--------+
|track_id              |popularity|danceability|energy|speechiness|acousticness|instrumentalness|liveness|valence|tempo  |loudness|duration_ms|key|mode|time_signature|explicit|
+----------------------+----------+------------+------+-----------+------------+----------------+--------+-------+-------+--------+-----------+---+----+--------------+--------+
|6rp6jA6hgJ97RIzil9ETzt|17.0      |0.779       |0.937 |0.0385     |0.0762      |2.17E-6         |0.776   |0.91   |101.873|-5.314  |280800.0   |2.0|0.0 |4.0           |0       |
|4rPmTBTX1wobDkGYXKvcGR|56.0      |0.606       |0.605 |0.0271     |0.118       |0.0             |0.194   |0.447  |133.883|-4.615  |352493.0   |6.0|1.0 |4.0           |0       |
|2AOmTB8IJm0decOoE21PxI|35.0      |0.68        |0.712 |0.0577     |7.07E-4     |0.

In [ ]:
#check for duplicate track IDs just in case, although already filtered when db created
duplicate_track_ids = (
    track_df
    .groupBy("track_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicate track IDs:", duplicate_track_ids)

track_df.select("popularity").describe().show()

Duplicate track IDs: 0
+-------+-----------------+
|summary|       popularity|
+-------+-----------------+
|  count|            89740|
|   mean|33.19990315939168|
| stddev|20.57525782781841|
|    min|              0.0|
|    max|            100.0|
+-------+-----------------+



# SPLIT DATA BEFORE TESTING 

80% of tracks will be used for training 
20% of tracks will be used for testing

In [16]:
train_raw, test_raw = track_df.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Training rows:", train_raw.count())
print("Test rows:", test_raw.count())

Training rows: 72027
Test rows: 17713


In [ ]:
# calculate threshold for 80th percentile of popularity in training set
threshold = train_raw.approxQuantile(
    "popularity",
    [0.80],
    0.001
)[0]

print("Training 80th percentile:", threshold)

Training 80th percentile: 52.0


In [ ]:
#create target for both datasets using training threshold
train_df = train_raw.withColumn(
    "hit",
    F.when(F.col("popularity") >= threshold, 1)
     .otherwise(0)
)

test_df = test_raw.withColumn(
    "hit",
    F.when(F.col("popularity") >= threshold, 1)
     .otherwise(0)
)

In [19]:
#check class distribution in training and test sets
print("TRAINING DISTRIBUTION")
train_df.groupBy("hit").count().orderBy("hit").show()

print("TEST DISTRIBUTION")
test_df.groupBy("hit").count().orderBy("hit").show()

TRAINING DISTRIBUTION
+---+-----+
|hit|count|
+---+-----+
|  0|56851|
|  1|15176|
+---+-----+

TEST DISTRIBUTION
+---+-----+
|hit|count|
+---+-----+
|  0|14131|
|  1| 3582|
+---+-----+



In [ ]:
#check for missing values in feature columns, there should be 0 becaue of the cleaning before creating the db
for c in feature_columns:
    missing = train_df.filter(
        F.col(c).isNull() |
        F.isnan(F.col(c).cast("double"))
    ).count()

    print(f"{c}: {missing} missing values")

danceability: 0 missing values
energy: 0 missing values
speechiness: 0 missing values
acousticness: 0 missing values
instrumentalness: 0 missing values
liveness: 0 missing values
valence: 0 missing values
tempo: 0 missing values
loudness: 0 missing values
duration_ms: 0 missing values
key: 0 missing values
mode: 0 missing values
time_signature: 0 missing values
explicit: 0 missing values


In [21]:
# create preprocessing pipeline, just in case although this shouldnt have to be used for feature columns: impute missing values, assemble into vector, and scale
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, VectorAssembler, StandardScaler

imputer = Imputer(
    inputCols=feature_columns,
    outputCols=[f"{c}_imputed" for c in feature_columns],
    strategy="median"
)

assembler = VectorAssembler(
    inputCols=[f"{c}_imputed" for c in feature_columns],
    outputCol="features",
    handleInvalid="error"
)

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withMean=False,
    withStd=True
)

# Training 3 models

In [22]:
# classifiers to be used
from pyspark.ml.classification import (
    LogisticRegression,
    RandomForestClassifier,
    GBTClassifier
)

In [23]:
#define models with hyperparameters/pipelines for each model
lr = LogisticRegression(
    labelCol="hit",
    featuresCol="scaled_features",
    maxIter=50,
    regParam=0.01
)

rf = RandomForestClassifier(
    labelCol="hit",
    featuresCol="features",
    numTrees=100,
    maxDepth=8,
    seed=42
)

gbt = GBTClassifier(
    labelCol="hit",
    featuresCol="features",
    maxIter=50,
    maxDepth=5,
    seed=42
)

lr_pipeline = Pipeline(
    stages=[imputer, assembler, scaler, lr]
)

rf_pipeline = Pipeline(
    stages=[imputer, assembler, rf]
)

gbt_pipeline = Pipeline(
    stages=[imputer, assembler, gbt]
)

In [24]:
#fit models to training data

lr_model = lr_pipeline.fit(train_df)

rf_model = rf_pipeline.fit(train_df)

gbt_model = gbt_pipeline.fit(train_df)

In [25]:
#see predictions on test data
lr_predictions = lr_model.transform(test_df)

rf_predictions = rf_model.transform(test_df)

gbt_predictions = gbt_model.transform(test_df)

In [26]:
lr_predictions.select(
    "track_id",
    "hit",
    "prediction",
    "probability"
).show(10, truncate=False)

+----------------------+---+----------+----------------------------------------+
|track_id              |hit|prediction|probability                             |
+----------------------+---+----------+----------------------------------------+
|001pyq8FLNSL1C8orNLI0b|0  |0.0       |[0.6995678951338995,0.3004321048661005] |
|00B7SBwrjbycLMOgAmeIU8|1  |0.0       |[0.7163386047487934,0.2836613952512066] |
|00C0DktT4pif6k7pVghYuI|0  |0.0       |[0.8636251132126741,0.1363748867873259] |
|00J8yszzb5PR6ZIvSBtg0Y|1  |0.0       |[0.7314122227944829,0.2685877772055171] |
|00Rptyyrh7YXjPwjd7SAsV|0  |0.0       |[0.8575722860502116,0.14242771394978837]|
|00Ys2R4wKqpKbGMgpZBirL|0  |0.0       |[0.8724859340445952,0.12751406595540482]|
|00oZhqZIQfL9P5CjOP6JsO|1  |0.0       |[0.7465346026743993,0.2534653973256007] |
|018tZLkNtsKAFDi2sqr2eX|0  |0.0       |[0.7258536643533849,0.27414633564661506]|
|01N4rGD2WJZewp4RAu8034|0  |0.0       |[0.8486362536857337,0.1513637463142663] |
|01Nl8C2RuzukJWbOfYxsU6|0  |

In [35]:
rf_predictions.select(
    "track_id",
    "hit",
    "prediction",
    "probability"
).show(10, truncate=False)

+----------------------+---+----------+----------------------------------------+
|track_id              |hit|prediction|probability                             |
+----------------------+---+----------+----------------------------------------+
|001pyq8FLNSL1C8orNLI0b|0  |0.0       |[0.7586483449624775,0.2413516550375225] |
|00B7SBwrjbycLMOgAmeIU8|1  |0.0       |[0.7413119492884277,0.2586880507115723] |
|00C0DktT4pif6k7pVghYuI|0  |0.0       |[0.8444072632290015,0.15559273677099852]|
|00J8yszzb5PR6ZIvSBtg0Y|1  |0.0       |[0.7434795081171979,0.25652049188280207]|
|00Rptyyrh7YXjPwjd7SAsV|0  |0.0       |[0.8031158695532131,0.1968841304467869] |
|00Ys2R4wKqpKbGMgpZBirL|0  |0.0       |[0.8538371497148598,0.14616285028514012]|
|00oZhqZIQfL9P5CjOP6JsO|1  |0.0       |[0.7553830611247966,0.24461693887520344]|
|018tZLkNtsKAFDi2sqr2eX|0  |0.0       |[0.7650880256742808,0.23491197432571914]|
|01N4rGD2WJZewp4RAu8034|0  |0.0       |[0.8142996443375057,0.18570035566249427]|
|01Nl8C2RuzukJWbOfYxsU6|0  |

In [36]:
gbt_predictions.select(
    "track_id",
    "hit",
    "prediction",
    "probability"
).show(10, truncate=False)

+----------------------+---+----------+----------------------------------------+
|track_id              |hit|prediction|probability                             |
+----------------------+---+----------+----------------------------------------+
|001pyq8FLNSL1C8orNLI0b|0  |0.0       |[0.751754319132821,0.24824568086717902] |
|00B7SBwrjbycLMOgAmeIU8|1  |0.0       |[0.6950753350479323,0.3049246649520677] |
|00C0DktT4pif6k7pVghYuI|0  |0.0       |[0.8800476370916672,0.11995236290833278]|
|00J8yszzb5PR6ZIvSBtg0Y|1  |0.0       |[0.640162927461781,0.35983707253821895] |
|00Rptyyrh7YXjPwjd7SAsV|0  |0.0       |[0.7838437768324553,0.21615622316754468]|
|00Ys2R4wKqpKbGMgpZBirL|0  |0.0       |[0.8376746284693092,0.16232537153069082]|
|00oZhqZIQfL9P5CjOP6JsO|1  |0.0       |[0.7053278798276293,0.29467212017237066]|
|018tZLkNtsKAFDi2sqr2eX|0  |0.0       |[0.6915382682307057,0.30846173176929426]|
|01N4rGD2WJZewp4RAu8034|0  |0.0       |[0.8610915076950112,0.1389084923049888] |
|01Nl8C2RuzukJWbOfYxsU6|0  |

# Evaluate models

Accuracy alone can be misleading when only around 20% of tracks are labelled as hits.

We'll measure:

Accuracy - Proportion of all predictions that are correct
Precision - Of the tracks predicted as hits, how many really are hits?
Recall- Of all actual hits, how many did the model identify?
F1-score - Balance between precision and recall
ROC-AUC - How well the model ranks positive cases above negative cases
PR-AUC -How well it identifies positive cases, especially useful with an imbalanced target

In [27]:
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator,
    BinaryClassificationEvaluator
)

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="hit",
    predictionCol="prediction",
    metricName="accuracy"
)

precision_evaluator = MulticlassClassificationEvaluator(
    labelCol="hit",
    predictionCol="prediction",
    metricName="precisionByLabel",
    metricLabel=1.0
)

recall_evaluator = MulticlassClassificationEvaluator(
    labelCol="hit",
    predictionCol="prediction",
    metricName="recallByLabel",
    metricLabel=1.0
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="hit",
    predictionCol="prediction",
    metricName="fMeasureByLabel",
    metricLabel=1.0,
    beta=1.0
)

roc_evaluator = BinaryClassificationEvaluator(
    labelCol="hit",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

pr_evaluator = BinaryClassificationEvaluator(
    labelCol="hit",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)

In [28]:
# evaluate model 
def evaluate_model(predictions):
    return {
        "accuracy": accuracy_evaluator.evaluate(predictions),
        "precision": precision_evaluator.evaluate(predictions),
        "recall": recall_evaluator.evaluate(predictions),
        "f1": f1_evaluator.evaluate(predictions),
        "roc_auc": roc_evaluator.evaluate(predictions),
        "pr_auc": pr_evaluator.evaluate(predictions)
    }

In [ ]:
#compare the 3 models based on evaluation metrics
results = {
    "Logistic Regression": evaluate_model(lr_predictions),
    "Random Forest": evaluate_model(rf_predictions),
    "Gradient-Boosted Trees": evaluate_model(gbt_predictions)
}

results_df = (
    spark.createDataFrame([
        (model_name, *metrics.values())
        for model_name, metrics in results.items()
    ], [
        "model",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc",
        "pr_auc"
    ])
    .orderBy(F.col("pr_auc").desc())
)

results_df.show(truncate=False)

+----------------------+------------------+------------------+--------------------+--------------------+------------------+-------------------+
|model                 |accuracy          |precision         |recall              |f1                  |roc_auc           |pr_auc             |
+----------------------+------------------+------------------+--------------------+--------------------+------------------+-------------------+
|Gradient-Boosted Trees|0.7979450121379777|0.5151515151515151|0.01423785594639866 |0.027709861450692742|0.6844900202188022|0.32574007630328633|
|Random Forest         |0.7977756450064925|0.0               |0.0                 |0.0                 |0.6674191177780883|0.3126780960890926 |
|Logistic Regression   |0.797606277875007 |0.3333333333333333|8.375209380234506E-4|0.001670843776106934|0.6289340497848539|0.2794746759891191 |
+----------------------+------------------+------------------+--------------------+--------------------+------------------+-------------

# Inspect errors

hit = 1 | prediction = 1 -> true positive
hit = 0 | prediction = 0 -> true negative
hit = 1 | prediction = 0 -> false negative
hit = 0 | prediction = 1 -> false positive

In [ ]:
# confusion matrix - random forest 
rf_predictions.groupBy(
    "hit",
    "prediction"
).count().orderBy(
    "hit",
    "prediction"
).show()

+---+----------+-----+
|hit|prediction|count|
+---+----------+-----+
|  0|       0.0|14131|
|  1|       0.0| 3582|
+---+----------+-----+



In [ ]:
# confusion matrix - logistic regression
lr_predictions.groupBy(
    "hit",
    "prediction"
).count().orderBy(
    "hit",
    "prediction"
).show()

+---+----------+-----+
|hit|prediction|count|
+---+----------+-----+
|  0|       0.0|14125|
|  0|       1.0|    6|
|  1|       0.0| 3579|
|  1|       1.0|    3|
+---+----------+-----+



In [ ]:
# confusion matrix - gradient boosted trees
gbt_predictions.groupBy(
    "hit",
    "prediction"
).count().orderBy(
    "hit",
    "prediction"
).show()

+---+----------+-----+
|hit|prediction|count|
+---+----------+-----+
|  0|       0.0|14083|
|  0|       1.0|   48|
|  1|       0.0| 3531|
|  1|       1.0|   51|
+---+----------+-----+



In [37]:
#indiv miscassifications for random forest
rf_predictions.filter(
    F.col("hit") != F.col("prediction")
).select(
    "track_id",
    "popularity",
    "hit",
    "prediction",
    "probability"
).show(20, truncate=False)

+----------------------+----------+---+----------+----------------------------------------+
|track_id              |popularity|hit|prediction|probability                             |
+----------------------+----------+---+----------+----------------------------------------+
|00B7SBwrjbycLMOgAmeIU8|66.0      |1  |0.0       |[0.7413119492884277,0.2586880507115723] |
|00J8yszzb5PR6ZIvSBtg0Y|66.0      |1  |0.0       |[0.7434795081171979,0.25652049188280207]|
|00oZhqZIQfL9P5CjOP6JsO|57.0      |1  |0.0       |[0.7553830611247966,0.24461693887520344]|
|01Rc6yXYtujAnMaR1NME11|71.0      |1  |0.0       |[0.7436496303226255,0.2563503696773745] |
|03fT3OHB9KyMtGMt2zwqCT|75.0      |1  |0.0       |[0.7465050377548509,0.25349496224514917]|
|04aulIHjjQyDVUUmaFDy8w|59.0      |1  |0.0       |[0.7122706891886476,0.28772931081135245]|
|05sqcYfU2wMlKwPVJ0rotq|58.0      |1  |0.0       |[0.7611171779516096,0.23888282204839043]|
|06PdA0DLgF4BfAeUNZAbFG|61.0      |1  |0.0       |[0.814981080653936,0.185018919

In [38]:
#indiv miscassifications for logistic regression
lr_predictions.filter(
    F.col("hit") != F.col("prediction")
).select(
    "track_id",
    "popularity",
    "hit",
    "prediction",
    "probability"
).show(20, truncate=False)

+----------------------+----------+---+----------+----------------------------------------+
|track_id              |popularity|hit|prediction|probability                             |
+----------------------+----------+---+----------+----------------------------------------+
|00B7SBwrjbycLMOgAmeIU8|66.0      |1  |0.0       |[0.7163386047487934,0.2836613952512066] |
|00J8yszzb5PR6ZIvSBtg0Y|66.0      |1  |0.0       |[0.7314122227944829,0.2685877772055171] |
|00oZhqZIQfL9P5CjOP6JsO|57.0      |1  |0.0       |[0.7465346026743993,0.2534653973256007] |
|01Rc6yXYtujAnMaR1NME11|71.0      |1  |0.0       |[0.8455817777321443,0.15441822226785573]|
|03fT3OHB9KyMtGMt2zwqCT|75.0      |1  |0.0       |[0.7869603201644556,0.2130396798355444] |
|04aulIHjjQyDVUUmaFDy8w|59.0      |1  |0.0       |[0.6807550589816722,0.3192449410183278] |
|05sqcYfU2wMlKwPVJ0rotq|58.0      |1  |0.0       |[0.8693898795498437,0.1306101204501563] |
|06PdA0DLgF4BfAeUNZAbFG|61.0      |1  |0.0       |[0.8309050462532925,0.16909495

In [39]:
#indiv miscassifications for gradient boosted trees
gbt_predictions.filter(
    F.col("hit") != F.col("prediction")
).select(
    "track_id",
    "popularity",
    "hit",
    "prediction",
    "probability"
).show(20, truncate=False)

+----------------------+----------+---+----------+----------------------------------------+
|track_id              |popularity|hit|prediction|probability                             |
+----------------------+----------+---+----------+----------------------------------------+
|00B7SBwrjbycLMOgAmeIU8|66.0      |1  |0.0       |[0.6950753350479323,0.3049246649520677] |
|00J8yszzb5PR6ZIvSBtg0Y|66.0      |1  |0.0       |[0.640162927461781,0.35983707253821895] |
|00oZhqZIQfL9P5CjOP6JsO|57.0      |1  |0.0       |[0.7053278798276293,0.29467212017237066]|
|01Rc6yXYtujAnMaR1NME11|71.0      |1  |0.0       |[0.6416846302840232,0.3583153697159768] |
|03fT3OHB9KyMtGMt2zwqCT|75.0      |1  |0.0       |[0.6928154044955778,0.30718459550442223]|
|04aulIHjjQyDVUUmaFDy8w|59.0      |1  |0.0       |[0.6643179009194549,0.33568209908054514]|
|05sqcYfU2wMlKwPVJ0rotq|58.0      |1  |0.0       |[0.7640244033535017,0.23597559664649825]|
|06PdA0DLgF4BfAeUNZAbFG|61.0      |1  |0.0       |[0.9187456750674846,0.08125432

# Select best model

PR-AUC as initial selection criterion

In a more complete project, we'd tune models using cross-validation on the training set and use the test set only for the final comparison.

In [42]:
best_model_name = max(
    results,
    key=lambda name: results[name]["pr_auc"]
)

print("Best model:", best_model_name)
print("Metrics:", results[best_model_name])

Best model: Gradient-Boosted Trees
Metrics: {'accuracy': 0.7979450121379777, 'precision': 0.5151515151515151, 'recall': 0.01423785594639866, 'f1': 0.027709861450692742, 'roc_auc': 0.6844900202188022, 'pr_auc': 0.32574007630328633}


# saving model